# Q4 — RNN cơ bản: Từ một bước thời gian đến cả chuỗi

CNN xử lý cấu trúc không gian. RNN xử lý dữ liệu có thứ tự theo thời gian hoặc theo chuỗi.

Một mạng nơ-ron hồi quy (**Recurrent Neural Network — RNN**) duy trì một **trạng thái ẩn
(trạng thái ẩn (hidden state))** tóm tắt thông tin từ các bước trước.

Với đầu vào $x_t$ và trạng thái trước $h_{t-1}$:

$$
h_t
=
\tanh(
x_tW_x+h_{t-1}W_h+b
).
$$

Trong notebook này, bạn sẽ:

- cài đặt một bước RNN;
- mở rộng thành cả chuỗi;
- quan sát gradient qua chuỗi dài;
- cài đặt word embedding;
- cài đặt lớp affine theo thời gian;
- cài đặt masked temporal softmax loss.

Ta dùng PyTorch tensor để autograd xử lý truyền ngược. Mục tiêu là hiểu forward computation
và cách các phần ghép với nhau.

In [ ]:
from pathlib import Path

assert Path("dl2").exists(), (
    "Không tìm thấy thư mục dl2/. "
    "Hãy chạy notebook từ thư mục gốc của Assignment 2."
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)

## 1. Một bước RNN

Kích thước:

- $x_t\in\mathbb{R}^{N\times D}$;
- $h_{t-1}\in\mathbb{R}^{N\times H}$;
- $W_x\in\mathbb{R}^{D\times H}$;
- $W_h\in\mathbb{R}^{H\times H}$;
- $b\in\mathbb{R}^{H}$.

Phép tính:

$$
a_t
=
x_tW_x+h_{t-1}W_h+b,
$$

$$
h_t
=
\tanh(a_t).
$$

Cùng $W_x,W_h,b$ được dùng lại ở **mọi bước thời gian**. Đây là cơ chế chia sẻ tham số
theo thời gian.

### Thực hành 1 — `rnn_step_forward`

In [ ]:
def rnn_step_forward(
    x,
    prev_h,
    Wx,
    Wh,
    b,
):
    """
    Tất cả đầu vào là torch.Tensor.

    Trả về
    -------
    next_h: (N, H)
    """
    # ============================================================
    # TODO: CODE CỦA BẠN
    #
    # Cài đặt đúng công thức vanilla RNN một bước.
    # Không dùng nn.RNNCell.
    # ============================================================
    raise NotImplementedError
    # ============================================================
    # KẾT THÚC CODE CỦA BẠN
    # ============================================================

### Kiểm tra với công thức trực tiếp

In [ ]:
N, D, H = 3, 4, 5

x = torch.randn(
    N,
    D,
    dtype=torch.float64,
)

prev_h = torch.randn(
    N,
    H,
    dtype=torch.float64,
)

Wx = torch.randn(
    D,
    H,
    dtype=torch.float64,
)

Wh = torch.randn(
    H,
    H,
    dtype=torch.float64,
)

b = torch.randn(
    H,
    dtype=torch.float64,
)

next_h = rnn_step_forward(
    x,
    prev_h,
    Wx,
    Wh,
    b,
)

expected = torch.tanh(
    x @ Wx
    + prev_h @ Wh
    + b
)

print(
    "Max abs error:",
    (next_h - expected)
    .abs()
    .max()
    .item()
)

assert torch.allclose(
    next_h,
    expected,
    atol=1e-12,
)

print("✓ rnn_step_forward đúng")

### Câu hỏi 1 — Chia sẻ tham số theo thời gian

Vì sao RNN dùng cùng $W_x$ và $W_h$ ở mọi timestep thay vì học một bộ tham số riêng cho
mỗi vị trí?

Nêu một lợi ích về số tham số và một lợi ích về khả năng xử lý độ dài chuỗi khác nhau.

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## 2. Từ một bước đến cả chuỗi

Giả sử:

$$
X\in\mathbb{R}^{N\times T\times D}.
$$

Ta xử lý lần lượt:

$$
h_1=f(x_1,h_0),
$$

$$
h_2=f(x_2,h_1),
$$

$$
\cdots
$$

$$
h_T=f(x_T,h_{T-1}).
$$

Đầu ra là toàn bộ trạng thái ẩn (hidden state)s:

$$
H\in\mathbb{R}^{N\times T\times H}.
$$

Không được dùng `nn.RNN` ở phần này. Hãy gọi lại `rnn_step_forward` trong vòng lặp.

### Thực hành 2 — `rnn_forward`

In [ ]:
def rnn_forward(
    x,
    h0,
    Wx,
    Wh,
    b,
):
    """
    x: (N, T, D)
    h0: (N, H)

    Trả về
    -------
    h: (N, T, H)
    """
    # ============================================================
    # TODO: CODE CỦA BẠN
    #
    # Duyệt theo T và dùng rnn_step_forward.
    # ============================================================
    raise NotImplementedError
    # ============================================================
    # KẾT THÚC CODE CỦA BẠN
    # ============================================================

### Kiểm tra kích thước và tính nhất quán

In [ ]:
N, T, D, H = 2, 6, 4, 5

x_seq = torch.randn(
    N,
    T,
    D,
    dtype=torch.float64,
)

h0 = torch.randn(
    N,
    H,
    dtype=torch.float64,
)

Wx = torch.randn(
    D,
    H,
    dtype=torch.float64,
)

Wh = torch.randn(
    H,
    H,
    dtype=torch.float64,
)

b = torch.randn(
    H,
    dtype=torch.float64,
)

h_seq = rnn_forward(
    x_seq,
    h0,
    Wx,
    Wh,
    b,
)

assert h_seq.shape == (
    N,
    T,
    H,
)

first_step = rnn_step_forward(
    x_seq[:, 0, :],
    h0,
    Wx,
    Wh,
    b,
)

assert torch.allclose(
    h_seq[:, 0, :],
    first_step,
)

print(
    "Hidden sequence shape:",
    h_seq.shape,
)
print("✓ rnn_forward hợp lệ")

## 3. Lan truyền ngược qua thời gian (Backpropagation Through Time — BPTT)

Khi loss ở bước cuối phụ thuộc vào nhiều trạng thái ẩn (hidden state)s trước đó, gradient phải đi ngược
qua chuỗi:

$$
h_T
\rightarrow
h_{T-1}
\rightarrow
\cdots
\rightarrow
h_0.
$$

Đây là **lan truyền ngược qua thời gian
(Backpropagation Through Time — BPTT)**.

Do cùng $W_h$ được dùng lặp lại, gradient chứa nhiều tích Jacobian liên tiếp.

Nếu các tích này có norm nhỏ hơn 1 nhiều lần, gradient có thể **tiêu biến
(vanishing gradient)**.

Nếu lớn hơn 1 nhiều lần, gradient có thể **bùng nổ (exploding gradient)**.

### Thí nghiệm — Đạo hàm và độ dài chuỗi

In [ ]:
def gradient_norm_for_length(
    T,
    recurrent_scale,
    seed=42,
):
    torch.manual_seed(seed)

    N, D, H = 1, 8, 16

    x = torch.randn(
        N,
        T,
        D,
        dtype=torch.float64,
    )

    h0 = torch.zeros(
        N,
        H,
        dtype=torch.float64,
        requires_grad=True,
    )

    Wx = (
        0.2
        * torch.randn(
            D,
            H,
            dtype=torch.float64,
        )
    )

    Wh = (
        recurrent_scale
        * torch.randn(
            H,
            H,
            dtype=torch.float64,
        )
        / np.sqrt(H)
    )

    b = torch.zeros(
        H,
        dtype=torch.float64,
    )

    h = rnn_forward(
        x,
        h0,
        Wx,
        Wh,
        b,
    )

    loss = (
        h[:, -1, :]
        .pow(2)
        .mean()
    )

    loss.backward()

    return h0.grad.norm().item()


lengths = [
    5,
    10,
    20,
    40,
    80,
]

# Sau khi rnn_forward hoàn thành, chạy hai trường hợp.
small_scale = [
    gradient_norm_for_length(
        T,
        recurrent_scale=0.5,
    )
    for T in lengths
]

large_scale = [
    gradient_norm_for_length(
        T,
        recurrent_scale=1.4,
    )
    for T in lengths
]

plt.semilogy(
    lengths,
    small_scale,
    marker="o",
    label="recurrent scale = 0.5",
)

plt.semilogy(
    lengths,
    large_scale,
    marker="o",
    label="recurrent scale = 1.4",
)

plt.xlabel(
    "Độ dài chuỗi T"
)
plt.ylabel(
    "||dL/dh0||"
)
plt.legend()
plt.grid(alpha=0.2)
plt.show()

### Câu hỏi 2 — Đạo hàm tiêu biến và đạo hàm bùng nổ

Dựa trên biểu đồ của bạn:

1. Gradient thay đổi như thế nào khi độ dài chuỗi tăng?
2. Vì sao scale của recurrent weights có thể ảnh hưởng mạnh đến gradient?
3. Gradient clipping giải quyết tốt hơn exploding hay vanishing gradient?

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## 4. Biểu diễn từ bằng vector nhúng (word embedding)

Trong bài toán ngôn ngữ, token thường được mã hóa bằng một số nguyên:

```text
"cat" -> 7
"dog" -> 12
```

Không nên đưa trực tiếp các số này vào RNN vì khoảng cách số học giữa ID không có ý nghĩa.

Ta học một ma trận embedding:

$$
W_{\text{embed}}
\in
\mathbb{R}^{V\times D},
$$

trong đó:

- $V$: kích thước từ vựng;
- $D$: số chiều embedding.

Token ID chỉ dùng để chọn một hàng của ma trận:

$$
e_t
=
W_{\text{embed}}[word_t].
$$

### Thực hành 3 — `word_embedding_forward`

In [ ]:
def word_embedding_forward(
    x,
    W_embed,
):
    """
    x: (N, T) chứa token IDs
    W_embed: (V, D)

    Trả về:
        out: (N, T, D)
    """
    # ============================================================
    # TODO: CODE CỦA BẠN
    #
    # Không dùng nn.Embedding trong phần này.
    # ============================================================
    raise NotImplementedError
    # ============================================================
    # KẾT THÚC CODE CỦA BẠN
    # ============================================================

In [ ]:
V, D = 12, 6

token_ids = torch.tensor([
    [1, 4, 2],
    [3, 3, 7],
])

W_embed = torch.randn(
    V,
    D,
)

embedded = word_embedding_forward(
    token_ids,
    W_embed,
)

assert embedded.shape == (
    2,
    3,
    D,
)

assert torch.allclose(
    embedded[0, 1],
    W_embed[4],
)

print(
    "Embedding shape:",
    embedded.shape,
)
print("✓ word_embedding_forward đúng")

### Câu hỏi 3 — Mã hóa one-hot và tra cứu embedding

Nếu biến mã token thành one-hot vector rồi nhân với $W_{embed}$, kết quả có giống
embedding tra cứu không?

Vì sao tra cứu thường hiệu quả hơn về bộ nhớ và tính toán?

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## 5. Lớp affine theo thời gian

RNN trả trạng thái ẩn (hidden state) ở mọi timestep:

$$
H\in\mathbb{R}^{N\times T\times H}.
$$

Để dự đoán một token ở mỗi bước, cùng một affine transform được dùng cho mọi timestep:

$$
S_{n,t}
=
H_{n,t}W+b.
$$

Nếu từ vựng (vocabulary) có $V$ từ:

$$
S
\in
\mathbb{R}^{N\times T\times V}.
$$

### Thực hành 4 — `temporal_affine_forward`

In [ ]:
def temporal_affine_forward(
    x,
    w,
    b,
):
    """
    x: (N, T, D)
    w: (D, M)
    b: (M,)

    Trả về:
        out: (N, T, M)
    """
    # ============================================================
    # TODO: CODE CỦA BẠN
    # ============================================================
    raise NotImplementedError
    # ============================================================
    # KẾT THÚC CODE CỦA BẠN
    # ============================================================

In [ ]:
N, T, D, M = 2, 4, 5, 7

x = torch.randn(
    N,
    T,
    D,
)

w = torch.randn(
    D,
    M,
)

b = torch.randn(
    M,
)

scores = temporal_affine_forward(
    x,
    w,
    b,
)

assert scores.shape == (
    N,
    T,
    M,
)

expected = (
    x.reshape(N * T, D)
    @ w
    + b
).reshape(N, T, M)

assert torch.allclose(
    scores,
    expected,
)

print("✓ temporal_affine_forward đúng")

## 6. Hàm mất mát Softmax theo thời gian và mặt nạ (mask)

Các câu/caption có độ dài khác nhau. Khi ghép thành một batch, chuỗi ngắn được thêm
`<NULL>` ở cuối.

Ví dụ:

```text
<START> this image shows cat <END> <NULL> <NULL>
<START> category is dog <END>    <NULL> <NULL> <NULL>
```

Không được tính loss tại các vị trí `<NULL>`.

Ta dùng mask:

$$
M_{n,t}
=
\begin{cases}
1 & \text{nếu vị trí hợp lệ}\\
0 & \text{nếu là phần đệm (padding)}
\end{cases}
$$

Loss:

$$
L
=
\frac{1}{N}
\sum_{n,t}
M_{n,t}
\left(
-\log p(y_{n,t})
\right).
$$

Lưu ý: ở đây ta **cộng theo thời gian và chia theo batch size**, giống cách thường dùng
trong mô hình chuỗi.

### Thực hành 5 — `temporal_softmax_loss`

In [ ]:
def temporal_softmax_loss(
    scores,
    y,
    mask,
):
    """
    scores: (N, T, V)
    y: (N, T)
    mask: (N, T), bool hoặc 0/1

    Trả về scalar loss.
    """
    # ============================================================
    # TODO: CODE CỦA BẠN
    #
    # Dùng Cross-Entropy ở từng timestep và bỏ các vị trí mask=0.
    # Có thể dùng torch.nn.functional.cross_entropy với
    # reduction="none".
    # ============================================================
    raise NotImplementedError
    # ============================================================
    # KẾT THÚC CODE CỦA BẠN
    # ============================================================

### Kiểm tra mặt nạ

In [ ]:
N, T, V = 2, 4, 5

scores = torch.randn(
    N,
    T,
    V,
    requires_grad=True,
)

y = torch.tensor([
    [1, 2, 3, 0],
    [2, 1, 0, 0],
])

mask = torch.tensor([
    [1, 1, 1, 0],
    [1, 1, 0, 0],
], dtype=torch.bool)

loss = temporal_softmax_loss(
    scores,
    y,
    mask,
)

# Thay logits tại các vị trí mask=0 bằng giá trị rất khác.
scores_changed = (
    scores.detach()
    .clone()
)

scores_changed[
    ~mask
] = 100.0

loss_changed = temporal_softmax_loss(
    scores_changed,
    y,
    mask,
)

print("Loss:", loss.item())
print(
    "Loss sau khi đổi padding logits:",
    loss_changed.item(),
)

assert torch.allclose(
    loss.detach(),
    loss_changed.detach(),
    atol=1e-6,
)

print("✓ Vị trí padding không ảnh hưởng loss")

### Câu hỏi 4 — Vì sao cần mặt nạ?

Nếu không mask các `<NULL>` token, mô hình sẽ được tối ưu để làm gì ở cuối các chuỗi ngắn?

Điều đó có thể làm sai lệch loss và quá trình học như thế nào?

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## 7. Ép theo nhãn đúng (teacher forcing)

Trong lúc huấn luyện mô hình sinh chuỗi, ta thường đưa **token đúng của bước trước** làm
đầu vào cho bước hiện tại.

Cách này gọi là **teacher forcing**.

Khi suy luận, token đúng không còn sẵn. Mô hình phải dùng chính token vừa dự đoán để tạo
bước tiếp theo.

Sự khác nhau này dẫn tới một vấn đề thường gọi là **exposure bias**: mô hình được huấn
luyện trên lịch sử đúng nhưng khi suy luận có thể phải tiếp tục từ chính lỗi của nó.

### Câu hỏi 5 — Huấn luyện và suy luận

Giải thích sự khác nhau giữa teacher forcing khi huấn luyện và autoregressive sampling
khi suy luận.

Vì sao một lỗi sớm trong chuỗi có thể ảnh hưởng nhiều bước tiếp theo?

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## Tóm tắt Q4

Bạn đã xây các thành phần chính của vanilla RNN:

- một bước hồi quy;
- forward qua cả chuỗi;
- word embedding;
- temporal affine;
- temporal softmax loss có mask.

Q5 sẽ ghép các thành phần ở mức mô hình để sinh một chuỗi mô tả đơn giản từ ảnh.